In [1]:
import numpy as np
import pandas as pd
from skimage.feature import hog
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score
import joblib

from collections import Counter
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [2]:
data = pd.read_csv("../archive/ascii_character_classification.csv", header=0).sample(frac=.05)

label_counts = Counter(data.iloc[:, 0])
print(label_counts)

Counter({0: 2546, 70: 274, 20: 271, 14: 266, 61: 265, 15: 261, 59: 261, 77: 259, 33: 259, 65: 259, 53: 256, 29: 256, 89: 254, 13: 254, 6: 253, 45: 251, 4: 251, 12: 249, 9: 248, 71: 247, 37: 247, 72: 245, 87: 244, 69: 244, 67: 244, 26: 242, 52: 242, 11: 242, 84: 241, 93: 241, 81: 240, 88: 240, 46: 239, 25: 239, 47: 239, 56: 238, 83: 238, 74: 238, 28: 238, 92: 237, 30: 237, 1: 237, 62: 237, 63: 237, 76: 237, 22: 236, 44: 236, 73: 236, 41: 236, 91: 236, 58: 236, 95: 235, 51: 235, 24: 235, 7: 234, 40: 234, 32: 234, 49: 234, 54: 234, 57: 233, 60: 233, 48: 233, 17: 232, 55: 232, 79: 232, 68: 232, 10: 231, 75: 231, 19: 230, 3: 230, 34: 229, 38: 228, 66: 228, 94: 227, 23: 226, 16: 226, 2: 225, 80: 225, 36: 224, 39: 224, 78: 223, 64: 222, 35: 222, 21: 221, 8: 221, 5: 221, 42: 220, 85: 220, 82: 216, 27: 216, 43: 215, 31: 215, 86: 213, 90: 209, 50: 208, 18: 193})


In [3]:
X = data.iloc[:, 1:].astype("float64")   # Features are all columns except the first one
y = data.iloc[:, 0].astype("float64")     # Labels are the first column
# Initialize the oversampler


# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.reset_index(drop=True, inplace=True)
y_train.reset_index(drop=True, inplace=True)
# Optionally, if you want to convert them back to pandas DataFrames:
train_data = pd.concat([y_train, X_train], axis=1)
test_data = pd.concat([y_test, X_test], axis=1)

In [4]:
clf = svm.SVC(kernel='linear')
clf.fit(X_train, y_train)
joblib.dump(clf, '../artifacts/svm_model.pkl')

['../artifacts/svm_model.pkl']

In [5]:
y_pred = clf.predict(X_train)
train_accuracy = accuracy_score(y_train, y_pred)
y_pred = clf.predict(X_test)
print("test Shape:", X_test.shape)

test_accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print(f"Train Accuracy: {train_accuracy*100:.4f}%")
print(f"Test Accuracy: {test_accuracy*100:.4f}%")
print(f"F1 Score: {f1*100:.4f}%")
print(f"Recall: {recall*100:.4f}%")

test Shape: (5000, 100)
Train Accuracy: 94.5350%
Test Accuracy: 93.4200%
F1 Score: 92.5783%
Recall: 93.4200%


In [6]:
def extract_hog_features(images):
    hog_features = []
    for image in images:
        image_reshaped = image.reshape((10, 10))
        features = hog(image_reshaped, pixels_per_cell=(2, 2), cells_per_block=(1, 1), feature_vector=True)
        hog_features.append(features)
    return np.array(hog_features)

X_hog = extract_hog_features(np.array(X))

X_train, X_test, y_train, y_test = train_test_split(X_hog, y, test_size=0.2, random_state=42)


In [7]:
clf = svm.SVC(kernel='linear')
clf.fit(X_train, y_train)
joblib.dump(clf, '../artifacts/svm_model_hog.pkl')

['../artifacts/svm_model_hog.pkl']

In [8]:
y_pred = clf.predict(X_train)
train_accuracy = accuracy_score(y_train, y_pred)
y_pred = clf.predict(X_test)
print("test Shape:", X_test.shape)

test_accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print(f"Train Accuracy: {train_accuracy*100:.4f}%")
print(f"Test Accuracy: {test_accuracy*100:.4f}%")
print(f"F1 Score: {f1*100:.4f}%")
print(f"Recall: {recall*100:.4f}%")

test Shape: (5000, 225)
Train Accuracy: 96.9650%
Test Accuracy: 94.2800%
F1 Score: 93.4484%
Recall: 94.2800%


In [9]:
import cv2

def extract_sift_features(images):
    sift = cv2.SIFT_create()
    sift_features = []
    
    for image in images:
        image_reshaped = image.reshape((10, 10)).astype(np.uint8)
        keypoints, descriptors = sift.detectAndCompute(image_reshaped, None)
        
        # If no keypoints are found, use a zero array of the same length as a typical descriptor
        if descriptors is None:
            descriptors = np.zeros((1, sift.descriptorSize()), dtype=np.float32)
        
        # Flatten descriptors and use them as features
        features = descriptors.flatten()
        sift_features.append(features)
    
    return np.array(sift_features)

# Extract SIFT features
X_sift = extract_sift_features(np.array(X))

In [10]:
X_sift = extract_sift_features(np.array(X))

# Since the number of features might vary, we need to ensure consistent feature vector size
# Here, we'll pad with zeros to the maximum descriptor length found
max_len = max(len(f) for f in X_sift)
X_sift = np.array([np.pad(f, (0, max_len - len(f)), 'constant') for f in X_sift])

# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X_sift, y, test_size=0.2, random_state=42)

In [11]:
clf = svm.SVC(kernel='linear')
clf.fit(X_train, y_train)
joblib.dump(clf, '../artifacts/svm_model_sift.pkl')

['../artifacts/svm_model_sift.pkl']

In [12]:
y_pred = clf.predict(X_train)
train_accuracy = accuracy_score(y_train, y_pred)
y_pred = clf.predict(X_test)
print("test Shape:", X_test.shape)

test_accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print(f"Train Accuracy: {train_accuracy*100:.4f}%")
print(f"Test Accuracy: {test_accuracy*100:.4f}%")
print(f"F1 Score: {f1*100:.4f}%")
print(f"Recall: {recall*100:.4f}%")

test Shape: (5000, 128)
Train Accuracy: 10.1050%
Test Accuracy: 10.5000%
F1 Score: 1.9955%
Recall: 10.5000%
